## Load cleaned billings dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../../data/interim/billings_cleaned.csv')
df.head()


,co_ref,renewal_month,sustainability_score,total_renewal_score_new,last_years_price,auto_renewal_score,status_scores,anchoring_score,tenure_scores,proforma_auto_renewal,...,tenure_group,last_renewal,last_band,last_total_net_paid,last_connections,anchor_group,renewal_year,is_first_year,has_discount,discount_pct
0,VT6174,2024-11-01,8.0,42.5,799.0,9,9,7.5,9.0,True,...,3,2023-11-01,Band B,664.0,1,1,2024,0,0,0.0
1,VD3828,2025-08-01,8.0,41.5,799.0,9,9,7.5,8.0,True,...,1,2025-08-09,Band C1,919.0,0,1,2025,1,0,0.0
2,DV8120,2025-03-01,8.0,33.0,799.0,8,0,7.5,9.5,True,...,4+,2024-03-01,Band C1,749.0,1,1,2025,0,0,0.0
3,EZ9894,2025-06-01,9.5,44.5,799.0,9,9,7.5,9.5,True,...,4+,2024-06-01,Band C1,749.0,1,1,2025,0,0,0.0
4,FA8957,2025-03-01,9.5,42.5,799.0,9,8,7.5,8.5,True,...,3,2024-03-01,Band C1,749.0,1,1,2025,0,0,0.0


## Creation of price features

In [ ]:
# Absolute price increase
df['price_increase_abs'] = df['starting_net'] - df['last_years_price']

# Percentage price increase
df['price_increase_pct'] = df['price_increase_abs'] / df['last_years_price']

# Flag: did price go up?
df['price_increased_flag'] = (df['price_increase_abs'] > 0).astype(int)

# Verify
df[['starting_net', 'last_years_price', 'price_increase_abs','price_increase_pct', 'price_increased_flag']].head()


,starting_net,last_years_price,price_increase_abs,price_increase_pct,price_increased_flag
0,919,799.0,120.0,0.150188,1
1,919,799.0,120.0,0.150188,1
2,799,799.0,0.0,0.000000,0
3,799,799.0,0.0,0.000000,0
4,799,799.0,0.0,0.000000,0


## Creating band movement features

In [ ]:
# Ordinal mapping for band
band_order = {
    'Band A': 1, 'Band B': 2,  'Band C1': 3, 'Band C2': 4,'Band D': 5, 'Band E': 6,  'Band F': 7,  'Band F1': 8,
    'Band F2': 9,'Band G': 10, 'Band H': 11, 'Band I': 12,'Band J': 13,'Group': 14
}

df['band_ordinal']      = df['band'].map(band_order)
df['last_band_ordinal'] = df['last_band'].map(band_order)

# Did the band change?
df['band_changed']    = (df['band'] != df['last_band']).astype(int)

# Did it go up or down?
df['band_upgraded']   = (df['band_ordinal'] > df['last_band_ordinal']).astype(int)
df['band_downgraded'] = (df['band_ordinal'] < df['last_band_ordinal']).astype(int)

# Verify
df[['band', 'last_band', 'band_changed', 'band_upgraded', 'band_downgraded']].head(10)


,band,last_band,band_changed,band_upgraded,band_downgraded
0,Band C1,Band B,1,1,0
1,Band C1,Band C1,0,0,0
2,Band C1,Band C1,0,0,0
3,Band C1,Band C1,0,0,0
4,Band C1,Band C1,0,0,0
5,Band C1,Band C1,0,0,0
6,Band C1,Band C1,0,0,0
7,Band C1,Band C1,0,0,0
8,Band C1,Band C1,0,0,0
9,Band C1,Band C1,0,0,0


## Ordinal encoding for columns with hierarchy

In [ ]:
# Tenure group
tenure_map = {'1': 1, '2': 2, '3': 3, '4+': 4}
df['tenure_group_ordinal'] = df['tenure_group'].map(tenure_map)

# Anchor group
anchor_map = {'Independent': 0, '1': 1, '2': 2, '3': 3, '4 to 9': 4, '10+': 5}
df['anchor_group_ordinal'] = df['anchor_group'].map(anchor_map)

# Membership status
membership_map = {'Non Member': 0, 'Member Only': 1, 'In Progress': 2, 'Accredited': 3}
df['membership_status_ordinal'] = df['proforma_membership_status'].map(membership_map)

# Verify
df[['tenure_group', 'tenure_group_ordinal','anchor_group', 'anchor_group_ordinal','proforma_membership_status', 'membership_status_ordinal']].head()


,tenure_group,tenure_group_ordinal,anchor_group,anchor_group_ordinal,proforma_membership_status,membership_status_ordinal
0,3,3,1,1,Accredited,3
1,1,1,1,1,Accredited,3
2,4+,4,1,1,Member Only,1
3,4+,4,1,1,Accredited,3
4,3,3,1,1,Accredited,3


## Binary Encoding

In [ ]:
# Auto renewal flag 
df['has_auto_renewal'] = df['current_auto_renewal_flag'].map({'y': 1, 'n': 0})

# World Pay token  
df['has_world_pay_token'] = df['current_world_pay_token'].map({'y': 1, 'n': 0})

# Proforma auto renewal  
df['proforma_auto_renewal'] = df['proforma_auto_renewal'].map({True: 1, False: 0, 'True': 1, 'False': 0})
df['proforma_world_pay_token'] = df['proforma_world_pay_token'].map({True: 1, False: 0, 'True': 1, 'False': 0})

# Verify
df[['has_auto_renewal', 'has_world_pay_token','proforma_auto_renewal', 'proforma_world_pay_token']].value_counts().reset_index()

,has_auto_renewal,has_world_pay_token,proforma_auto_renewal,proforma_world_pay_token,count
0,1,0,1,0,52311
1,1,1,1,1,41889
2,1,1,1,0,16065
3,0,0,1,1,4631
4,1,0,1,1,2912
5,0,0,1,0,1649
6,0,0,0,0,983
7,1,0,0,0,945
8,1,1,0,0,545
9,1,1,0,1,10


## One hot encoding for categorical columns

In [ ]:
ohe_cols = ['payment_method', 'proforma_account_stage']

df = pd.get_dummies(df, columns=ohe_cols, drop_first=False)

# Verify new columns created
new_cols = [c for c in df.columns if 'payment_method' in c or 'proforma_account_stage' in c]
print(new_cols)

['payment_method_Bacs', 'payment_method_Card', 'payment_method_Cheque', 'payment_method_Unknown', 'payment_method_World Pay', 'proforma_account_stage_Membership Only', 'proforma_account_stage_Published', 'proforma_account_stage_Renewal Process', 'proforma_account_stage_Retired', 'proforma_account_stage_Suspended', 'proforma_account_stage_Unknown', 'proforma_account_stage_Vetting']


## Creating churn label

In [ ]:
df = df[df['prospect_outcome'].isin(['Churned', 'Won'])].copy()
df['churn'] = (df['prospect_outcome'] == 'Churned').astype(int)
df['churn'].value_counts()

churn
0    101172
1     12597
Name: count, dtype: int64

## Dropping columns no longer needed

In [ ]:
drop_cols = [
    'current_auto_renewal_flag',  # replaced by has_auto_renewal
    'current_world_pay_token',   # replaced by has_world_pay_token
    'proforma_membership_status',  # replaced by membership_status_ordinal
    'tenure_group',      # replaced by tenure_group_ordinal
    'anchor_group',       # replaced by anchor_group_ordinal
    'band',           # replaced by band_ordinal
    'last_band',         # replaced by last_band_ordinal
    'prospect_outcome',      # replaced by churn label
]

df.drop(columns=drop_cols, inplace=True)


## Saving final features dataset

In [ ]:
df.to_csv('../../../data/interim/billings_features.csv', index=False)